# 微调


在上一节中，我们学习了如何正确构建训练和测试数据。在本教程中，我们将实际进行模型微调。



## 安装

注意，要使用 FlagEmbedding 微调 BGE 模型，我们需要安装带有 finetune 依赖的软件包：

In [ ]:
% pip install -U FlagEmbedding[finetune]

## Fine-tune

以下是微调的参数：

以下参数用于模型：
- `model_name_or_path`：初始化使用的模型检查点。
- `config_name`：预训练配置名称或路径（如果与model_name不同）。
- `tokenizer_name`：预训练分词器名称或路径（如果与model_name不同）。
- `cache_dir`：存储从s3下载的预训练模型的位置。
- `trust_remote_code`：是否信任远程代码。
- `token`：访问模型时使用的令牌。

以下参数用于数据：
- `train_data`：训练数据的一个或多个路径。训练数据中需要包含`query: str`、`pos: List[str]`、`neg: List[str]`。参数类型：多个。
- `cache_path`：存储缓存数据的位置。
- `train_group_size`：（未提供元数据）
- `query_max_len`：查询经过分词后的最大总输入序列长度。超过此长度的序列将被截断。
- `passage_max_len`：段落经过分词后的最大总输入序列长度。超过此长度的序列将被截断。
- `pad_to_multiple_of`：如果设置，将把序列填充为提供值的倍数。
- `max_example_num_per_dataset`：每个数据集的最大样本数。
- `query_instruction_for_retrieval`：查询的指令。
- `query_instruction_format`：查询指令的格式。
- `knowledge_distillation`：当训练数据的特征中包含`pos_scores: List[float]`和`neg_scores: List[float]`时使用知识蒸馏。
- `passage_instruction_for_retrieval`：段落的指令。
- `passage_instruction_format`：段落指令的格式。
- `shuffle_ratio`：文本打乱的比例。
- `same_dataset_within_batch`：同一批次中的所有样本来自同一数据集。
- `small_threshold`：小数据集的阈值。同一目录下的所有小数据集将被合并为一个数据集。
- `drop_threshold`：丢弃合并小数据集的阈值。如果合并小数据集中的样本数小于此阈值，它将被丢弃。

以及以下额外参数：
- `negatives_cross_device`：跨设备共享负样本。
- `temperature`：用于相似度分数的温度参数。
- `fix_position_embedding`：冻结位置嵌入的参数。
- `sentence_pooling_method`：池化方法。可用选项：cls、mean、last_token。默认值：cls。
- `normalize_embeddings`：是否对嵌入进行归一化。
- `sub_batch_size`：训练的子批次大小。
- `kd_loss_type`：知识蒸馏的损失类型。可用选项：kl_div、m3_kd_loss。默认值：kl_div。

In [ ]:
%%bash
torchrun --nproc_per_node 2 \
	-m FlagEmbedding.finetune.embedder.encoder_only.base \
	--model_name_or_path BAAI/bge-large-en-v1.5 \
    --cache_dir ./cache/model \
    --train_data ./ft_data/training.json \
    --cache_path ./cache/data \
    --train_group_size 8 \
    --query_max_len 512 \
    --passage_max_len 512 \
    --pad_to_multiple_of 8 \
    --query_instruction_for_retrieval 'Represent this sentence for searching relevant passages: ' \
    --query_instruction_format '{}{}' \
    --knowledge_distillation False \
	--output_dir ./test_encoder_only_base_bge-large-en-v1.5 \
    --overwrite_output_dir \
    --learning_rate 1e-5 \
    --fp16 \
    --num_train_epochs 2 \
    --per_device_train_batch_size 2 \
    --dataloader_drop_last True \
    --warmup_ratio 0.1 \
    --gradient_checkpointing \
    --deepspeed config/ds_stage0.json \
    --logging_steps 1 \
    --save_steps 1000 \
    --negatives_cross_device \
    --temperature 0.02 \
    --sentence_pooling_method cls \
    --normalize_embeddings True \
    --kd_loss_type kl_div